In [14]:
import json
import random
from datetime import datetime
import pandas as pd

# Load car brands and models from JSON
file_path = 'car_brands_and_models.json'
with open(file_path, 'r') as file:
    car_data = json.load(file)

# Generate a random month/year in the format MM/YYYY
def generate_month_year(start_year, end_year):
    year = random.randint(start_year, end_year)
    month = random.randint(1, 12)
    return f"{month:02d}/{year}"

# Generate a dataset of used cars
def generate_used_cars(num_records, file_format):
    records = []
    current_year = datetime.now().year
    fuel_types = ['benzine', 'diesel', 'lpg', 'elektrisch']

    for _ in range(num_records):
        brand = random.choice(list(car_data.keys()))
        model = random.choice(car_data[brand])
        cilinderinhoud = random.randint(1000, 3600)
        cilinderinhoud = round(cilinderinhoud / 100) * 100
        bouwjaar = generate_month_year(1990, current_year)
        brandstof = random.choice(fuel_types)
        kilometerstand = random.randint(0, 300_000)
        vermogen_kw = random.randint(40, 400)
        vermogen_pk = round(vermogen_kw * 1.341)

        records.append({
            "Merk": brand,
            "Model": model,
            "Cilinderinhoud (cc)": cilinderinhoud,
            "Bouwjaar": bouwjaar,
            "Brandstof": brandstof,
            "Kilometerstand": kilometerstand,
            "Vermogen (kW)": vermogen_kw,
            "Vermogen (pk)": vermogen_pk
        })

    # Save to file
    if file_format == 'csv':
        df = pd.DataFrame(records)
        file_name = 'generated/used_cars_dataset.csv'
        df.to_csv(file_name, index=False)
    elif file_format == 'json':
        file_name = 'generated/used_cars_dataset.json'
        with open(file_name, 'w') as json_file:
            json.dump(records, json_file, indent=4)

    return file_name

# Introduce errors into the dataset
def introduce_errors(file_path, fraction):
    with open(file_path, 'r') as file:
        data = json.load(file) if file_path.endswith('.json') else pd.read_csv(file_path).to_dict('records')

    num_errors = int(len(data) * fraction)
    error_indices = random.sample(range(len(data)), num_errors)
    error_types = [
        lambda r: r.update({"Merk": ''.join(random.choice('abcdefghijklmnopqrstuvwxyz') if i == random.randint(0, len(r["Merk"]) - 1) else c for i, c in enumerate(r["Merk"]))}),
        lambda r: r.update({"Model": ''.join(random.choice('abcdefghijklmnopqrstuvwxyz') if i == random.randint(0, len(r["Model"]) - 1) else c for i, c in enumerate(r["Model"]))}),
        lambda r: r.update({"Model": "Warburg", "Model": "Rumplumplum"}),
        lambda r: r.update({"Kilometerstand": -1}),
        lambda r: r.update({"Kilometerstand": 1_000_010}),
        lambda r: r.update({"Cilinderinhoud (cc)": 4800}),
        lambda r: r.update({"Bouwjaar": generate_month_year(1940, 1989)}),
        lambda r: r.update({"Brandstof": "biodiesel"})
    ]

    for index in error_indices:
        random.choice(error_types)(data[index])

    # Save the modified dataset
    if file_path.endswith('.csv'):
        df = pd.DataFrame(data)
        modified_file_path = 'generated/modified_used_cars_dataset.csv'
        df.to_csv(modified_file_path, index=False)
    elif file_path.endswith('.json'):
        modified_file_path = 'generated/modified_used_cars_dataset.json'
        with open(modified_file_path, 'w') as file:
            json.dump(data, file, indent=4)

    return modified_file_path

# Example: Generate dataset and introduce errors
file_path = generate_used_cars(5000, 'csv')
modified_file_path = introduce_errors(file_path, 0.2)
modified_file_path


'generated/modified_used_cars_dataset.csv'